# v1.1 Crime Data Quality Audit

This notebook establishes the data-quality and classification rules that will
support the Seattle Public Safety Dashboard v1.1 analytical metrics.

## Research questions

1. Are unmappable crimes included in non-geographic analytical totals and the time series?
2. What exactly does subtype/code `999` represent?
3. What records currently fall into the `other` crime category?
4. Which `other` records should be retained, reclassified, or excluded?
5. Are there additional null, duplicate, administrative, or offense-code behaviors
   that could distort future KPI calculations?

No production classification rules should be changed until the evidence in this
notebook has been reviewed.

In [ ]:
from pathlib import Path
import sys
import re

import numpy as np
import pandas as pd
from IPython.display import display


# -------------------------------------------------------------------
# Locate repository root robustly whether Jupyter starts from repo root
# or from the notebooks directory.
# -------------------------------------------------------------------

cwd = Path.cwd().resolve()

repo_candidates = [cwd, *cwd.parents]

REPO_ROOT = next(
    (
        path
        for path in repo_candidates
        if (path / "dashboard").is_dir()
        and (path / "app.py").exists()
    ),
    None,
)

if REPO_ROOT is None:
    raise RuntimeError(
        "Could not locate repository root. "
        "Expected to find app.py and dashboard/."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository root: {REPO_ROOT}")


# -------------------------------------------------------------------
# Dashboard imports
# -------------------------------------------------------------------

from dashboard.crime_dashboard_data import (
    load_crime_dashboard_context,
    EVENT_ID_COLUMN,
    ROW_ID_COLUMN,
    TIME_COLUMN,
    REPORT_TIME_COLUMN,
    LAT_COL,
    LON_COL,
    CATEGORY_COLUMN,
    SUB_CATEGORY_COLUMN,
    NEIGHBORHOOD_COLUMN,
)

from dashboard.crime_dashboard_figures import (
    TARGET_CRIME_CATEGORIES,
    prepare_daily_event_data,
)

from dashboard.crime_filters import filter_crime_records


# -------------------------------------------------------------------
# Display configuration
# -------------------------------------------------------------------

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 100)


# -------------------------------------------------------------------
# Load the same context used by the dashboard
# -------------------------------------------------------------------

context = load_crime_dashboard_context()

crime = context["df"].copy()
valid_time = context["valid_time"].copy()
mappable_events = context["mappable_events"].copy()
unmappable_events = context["unmappable_events"].copy()


# Normalize key fields for this audit.
crime[TIME_COLUMN] = pd.to_datetime(
    crime[TIME_COLUMN],
    errors="coerce",
)

crime[REPORT_TIME_COLUMN] = pd.to_datetime(
    crime[REPORT_TIME_COLUMN],
    errors="coerce",
)

valid_time[TIME_COLUMN] = pd.to_datetime(
    valid_time[TIME_COLUMN],
    errors="coerce",
)

valid_time["date"] = valid_time[TIME_COLUMN].dt.normalize()


def clean_string(series: pd.Series) -> pd.Series:
    return (
        series
        .astype("string")
        .str.strip()
        .str.lower()
    )


def valid_string_mask(series: pd.Series) -> pd.Series:
    cleaned = clean_string(series)

    return (
        cleaned.notna()
        & ~cleaned.isin(
            {
                "",
                "-",
                "unknown",
                "none",
                "nan",
                "<na>",
            }
        )
    )


print()
print("Crime snapshot")
print("------------------------------")
print(f"Rows:             {len(crime):,}")
print(
    f"Unique offenses:  "
    f"{crime[EVENT_ID_COLUMN].nunique(dropna=True):,}"
)
print(
    f"Valid-time rows:  "
    f"{len(valid_time):,}"
)
print(
    f"Valid-time IDs:   "
    f"{valid_time[EVENT_ID_COLUMN].nunique(dropna=True):,}"
)
print(
    f"Date range:       "
    f"{valid_time[TIME_COLUMN].min():%Y-%m-%d}"
    f" to "
    f"{valid_time[TIME_COLUMN].max():%Y-%m-%d}"
)
print()
print("Dashboard crime categories:")
print(TARGET_CRIME_CATEGORIES)

Repository root: C:\Users\benca\code\PersonalPythonProjects\SPDCallDashboard

Crime snapshot
------------------------------
Rows:             76,139
Unique offenses:  76,139
Valid-time rows:  76,139
Valid-time IDs:   76,139
Date range:       2025-09-06 to 2026-09-06

Dashboard crime categories:
['other (includes drug and sex offenses)', 'property crime', 'violent crime']


## 1. Unmappable-event inclusion

We need to distinguish two concepts:

- **Analytically valid offense:** has the identifiers/timestamps needed for analysis.
- **Mappable offense:** additionally has usable geographic coordinates.

Coordinate validity should affect point mapping, but it should not remove an
otherwise valid offense from citywide totals or daily crime counts.

This section compares the dashboard's actual daily-series output against counts
computed directly from `valid_time`.

In [ ]:
# -------------------------------------------------------------------
# Establish which analytically valid offenses are mappable.
# -------------------------------------------------------------------

valid_time[EVENT_ID_COLUMN] = clean_string(
    valid_time[EVENT_ID_COLUMN]
)

mappable_ids = set(
    clean_string(mappable_events[EVENT_ID_COLUMN])
    .dropna()
    .tolist()
)

valid_time["is_mappable"] = (
    valid_time[EVENT_ID_COLUMN].isin(mappable_ids)
)

total_valid_offenses = valid_time[EVENT_ID_COLUMN].nunique()

mappable_offense_count = (
    valid_time.loc[
        valid_time["is_mappable"],
        EVENT_ID_COLUMN,
    ]
    .nunique()
)

unmappable_offense_count = (
    valid_time.loc[
        ~valid_time["is_mappable"],
        EVENT_ID_COLUMN,
    ]
    .nunique()
)

unmappable_pct = (
    100 * unmappable_offense_count / total_valid_offenses
    if total_valid_offenses
    else np.nan
)


# -------------------------------------------------------------------
# Ask the actual dashboard preparation function for its daily counts.
# -------------------------------------------------------------------

daily_chart, daily_window = prepare_daily_event_data(
    context=context,
    selected_bins=TARGET_CRIME_CATEGORIES,
    analysis_state=None,
)

plot_start = pd.Timestamp(
    daily_window["plot_start_day"]
).normalize()

plot_end = pd.Timestamp(
    daily_window["plot_end_day"]
).normalize()

chart_dates = pd.to_datetime(
    daily_chart["date"]
).dt.normalize()


# -------------------------------------------------------------------
# Independently reconstruct what the daily series SHOULD contain
# from valid_time.
# -------------------------------------------------------------------

eligible_for_chart = valid_time[
    valid_time["date"].between(
        plot_start,
        plot_end,
    )
    & valid_time["event_importance_bin"].isin(
        TARGET_CRIME_CATEGORIES
    )
].copy()

expected_daily = (
    eligible_for_chart
    .groupby("date")[EVENT_ID_COLUMN]
    .nunique()
    .reindex(chart_dates, fill_value=0)
    .astype(int)
)

actual_daily = (
    daily_chart
    .set_index(pd.to_datetime(daily_chart["date"]).dt.normalize())
    ["reported_offenses"]
    .astype(int)
)

daily_comparison = pd.DataFrame(
    {
        "expected_from_valid_time": expected_daily,
        "dashboard_daily_count": actual_daily,
    }
)

daily_comparison["difference"] = (
    daily_comparison["dashboard_daily_count"]
    - daily_comparison["expected_from_valid_time"]
)

mismatch_days = daily_comparison[
    daily_comparison["difference"] != 0
]


# -------------------------------------------------------------------
# Verify the unified citywide analytical filter also retains
# unmappable events when no neighborhood filter is selected.
# -------------------------------------------------------------------

citywide_state = {
    "start_date": plot_start.strftime("%Y-%m-%d"),
    "end_date": plot_end.strftime("%Y-%m-%d"),
    "crime_categories": TARGET_CRIME_CATEGORIES,
    "crime_subcategories": [],
    "neighborhoods": [],
}

citywide_filtered = filter_crime_records(
    valid_time,
    citywide_state,
)

expected_citywide_ids = set(
    eligible_for_chart[EVENT_ID_COLUMN]
    .dropna()
    .unique()
)

actual_citywide_ids = set(
    citywide_filtered[EVENT_ID_COLUMN]
    .dropna()
    .unique()
)

expected_unmappable_ids = (
    expected_citywide_ids - mappable_ids
)

actual_unmappable_ids = (
    actual_citywide_ids - mappable_ids
)


# -------------------------------------------------------------------
# Check the special first-day behavior in the current daily window.
# -------------------------------------------------------------------

earliest_available_day = pd.Timestamp(
    daily_window["earliest_available_day"]
).normalize()

earliest_day_offenses = (
    valid_time.loc[
        valid_time["date"].eq(earliest_available_day)
        & valid_time["event_importance_bin"].isin(
            TARGET_CRIME_CATEGORIES
        ),
        EVENT_ID_COLUMN,
    ]
    .nunique()
)


summary = pd.DataFrame(
    {
        "metric": [
            "Analytically valid unique offenses",
            "Mappable unique offenses",
            "Unmappable unique offenses",
            "Percent unmappable",
            "Daily-chart mismatch days",
            "Expected citywide offenses in chart window",
            "Actual citywide offenses after shared filter",
            "Expected unmappable offenses in chart window",
            "Unmappable offenses retained by shared filter",
            "Offenses on earliest available day",
        ],
        "value": [
            total_valid_offenses,
            mappable_offense_count,
            unmappable_offense_count,
            round(unmappable_pct, 2),
            len(mismatch_days),
            len(expected_citywide_ids),
            len(actual_citywide_ids),
            len(expected_unmappable_ids),
            len(actual_unmappable_ids),
            earliest_day_offenses,
        ],
    }
)

display(summary)

print()

if mismatch_days.empty:
    print(
        "PASS: Daily dashboard counts exactly match valid_time "
        "within the configured chart window."
    )
else:
    print(
        f"REVIEW: {len(mismatch_days):,} date(s) differ between "
        "valid_time and the dashboard daily series."
    )
    display(mismatch_days)

if expected_unmappable_ids == actual_unmappable_ids:
    print(
        "PASS: Unmappable offenses are retained by the citywide "
        "shared analytical filter."
    )
else:
    missing = expected_unmappable_ids - actual_unmappable_ids

    print(
        "REVIEW: Some unmappable offenses disappear after "
        "citywide filtering."
    )
    print(f"Missing unmappable offense IDs: {len(missing):,}")

print()

if plot_start > earliest_available_day:
    print(
        "IMPORTANT WINDOW NOTE:"
    )
    print(
        f"The daily chart's configured analysis window begins "
        f"{plot_start.date()}, while the earliest available date is "
        f"{earliest_available_day.date()}."
    )
    print(
        f"{earliest_day_offenses:,} unique offense(s) occur on that "
        "earliest date."
    )

,metric,value
0,Analytically valid unique offenses,76139.00
1,Mappable unique offenses,62772.00
2,Unmappable unique offenses,13367.00
3,Percent unmappable,17.56
4,Daily-chart mismatch days,0.00
5,Expected citywide offenses in chart window,76109.00
6,Actual citywide offenses after shared filter,76109.00
7,Expected unmappable offenses in chart window,13363.00
8,Unmappable offenses retained by shared filter,13363.00
9,Offenses on earliest available day,30.00



PASS: Daily dashboard counts exactly match valid_time within the configured chart window.
PASS: Unmappable offenses are retained by the citywide shared analytical filter.

IMPORTANT WINDOW NOTE:
The daily chart's configured analysis window begins 2025-09-07, while the earliest available date is 2025-09-06.
30 unique offense(s) occur on that earliest date.


IMPORTANT NOTE - It seems nearly a fifth of all observations are unmappable, this warrants some annotating on the point map.

## 2. Investigation of subtype/code `999`

Rather than assuming that `999` means "not a crime", search every relevant
classification field for the value and inspect the categories, NIBRS codes,
descriptions, and crime-against classifications associated with those records.

We will also compare `999` records against text explicitly indicating
"not a crime", "non-criminal", or similar terminology.

In [ ]:
CLASSIFICATION_COLUMNS = [
    CATEGORY_COLUMN,
    SUB_CATEGORY_COLUMN,
    "nibrs_offense_code",
    "nibrs_offense_code_description",
    "nibrs_group_a_b",
    "nibrs_crime_against_category",
]

CLASSIFICATION_COLUMNS = [
    column
    for column in CLASSIFICATION_COLUMNS
    if column in crime.columns
]


# -------------------------------------------------------------------
# Search all classification columns for 999.
# -------------------------------------------------------------------

code_999_summary = []

match_masks = {}

for column in CLASSIFICATION_COLUMNS:
    values = clean_string(crime[column])

    exact_999 = values.eq("999")

    contains_999 = values.str.contains(
        r"(?<!\d)999(?!\d)",
        regex=True,
        na=False,
    )

    match_masks[column] = contains_999

    code_999_summary.append(
        {
            "column": column,
            "exact_999_rows": int(exact_999.sum()),
            "contains_999_rows": int(contains_999.sum()),
            "unique_offenses": int(
                crime.loc[
                    contains_999,
                    EVENT_ID_COLUMN,
                ].nunique()
            ),
        }
    )

code_999_summary = pd.DataFrame(
    code_999_summary
).sort_values(
    "contains_999_rows",
    ascending=False,
)

display(code_999_summary)


# -------------------------------------------------------------------
# Combine all 999-related records.
# -------------------------------------------------------------------

any_999_mask = pd.Series(
    False,
    index=crime.index,
)

for mask in match_masks.values():
    any_999_mask |= mask

records_999 = crime[
    any_999_mask
].copy()

print(
    f"Rows matching 999 anywhere: {len(records_999):,}"
)

print(
    f"Unique offenses matching 999: "
    f"{records_999[EVENT_ID_COLUMN].nunique():,}"
)


# -------------------------------------------------------------------
# Show the semantic combinations attached to 999.
# -------------------------------------------------------------------

detail_columns = [
    CATEGORY_COLUMN,
    SUB_CATEGORY_COLUMN,
    "nibrs_offense_code",
    "nibrs_offense_code_description",
    "nibrs_group_a_b",
    "nibrs_crime_against_category",
]

detail_columns = [
    column
    for column in detail_columns
    if column in records_999.columns
]

if not records_999.empty:
    code_999_detail = (
        records_999
        .groupby(
            detail_columns,
            dropna=False,
        )
        .agg(
            rows=(EVENT_ID_COLUMN, "size"),
            unique_offenses=(
                EVENT_ID_COLUMN,
                "nunique",
            ),
        )
        .reset_index()
        .sort_values(
            "unique_offenses",
            ascending=False,
        )
    )

    display(code_999_detail)
else:
    print("No 999 records were found.")

,column,exact_999_rows,contains_999_rows,unique_offenses
1,offense_sub_category,9088,9088,9088
2,nibrs_offense_code,9088,9088,9088
0,offense_category,0,0,0
3,nibrs_offense_code_description,0,0,0
4,nibrs_group_a_b,0,0,0
5,nibrs_crime_against_category,0,0,0


Rows matching 999 anywhere: 9,088
Unique offenses matching 999: 9,088


,offense_category,offense_sub_category,nibrs_offense_code,nibrs_offense_code_description,nibrs_group_a_b,nibrs_crime_against_category,rows,unique_offenses
0,other (includes drug and sex offenses),999,999,not reportable to nibrs,b,not_a_crime,9088,9088


In [ ]:
NON_CRIME_PATTERN = (
    r"not[\s_-]+(?:a[\s_-]+)?crime"
    r"|non[\s_-]?criminal"
    r"|no[\s_-]+crime"
    r"|administrative"
)

semantic_text_columns = [
    SUB_CATEGORY_COLUMN,
    "nibrs_offense_code_description",
    "nibrs_crime_against_category",
]

semantic_text_columns = [
    column
    for column in semantic_text_columns
    if column in crime.columns
]

non_crime_mask = pd.Series(
    False,
    index=crime.index,
)

for column in semantic_text_columns:
    non_crime_mask |= (
        clean_string(crime[column])
        .str.contains(
            NON_CRIME_PATTERN,
            regex=True,
            na=False,
        )
    )

comparison_999_noncrime = pd.DataFrame(
    {
        "classification": [
            "999-related",
            "Explicit non-crime/admin text",
            "Both 999 and non-crime/admin text",
            "999 without explicit non-crime/admin text",
        ],
        "unique_offenses": [
            crime.loc[
                any_999_mask,
                EVENT_ID_COLUMN,
            ].nunique(),
            crime.loc[
                non_crime_mask,
                EVENT_ID_COLUMN,
            ].nunique(),
            crime.loc[
                any_999_mask & non_crime_mask,
                EVENT_ID_COLUMN,
            ].nunique(),
            crime.loc[
                any_999_mask & ~non_crime_mask,
                EVENT_ID_COLUMN,
            ].nunique(),
        ],
    }
)

display(comparison_999_noncrime)

if "nibrs_crime_against_category" in crime.columns:
    display(
        crime.loc[
            clean_string(
                crime["nibrs_crime_against_category"]
            ).str.contains(
                NON_CRIME_PATTERN,
                regex=True,
                na=False,
            ),
            "nibrs_crime_against_category",
        ]
        .value_counts(dropna=False)
        .rename("count")
        .to_frame()
    )

if not records_999.empty:
    columns_to_show = [
        CATEGORY_COLUMN,
        SUB_CATEGORY_COLUMN,
        "nibrs_offense_code",
        "nibrs_offense_code_description",
        "nibrs_group_a_b",
        "nibrs_crime_against_category",
    ]

    columns_to_show = [
        column
        for column in columns_to_show
        if column in records_999.columns
    ]

    print("Distinct classifications attached to 999:")
    display(
        records_999[
            columns_to_show
        ]
        .drop_duplicates()
        .sort_values(
            columns_to_show,
            na_position="last",
        )
    )

,classification,unique_offenses
0,999-related,9088
1,Explicit non-crime/admin text,9091
2,Both 999 and non-crime/admin text,9088
3,999 without explicit non-crime/admin text,0


,count
nibrs_crime_against_category,
not_a_crime,9091


Distinct classifications attached to 999:


,offense_category,offense_sub_category,nibrs_offense_code,nibrs_offense_code_description,nibrs_group_a_b,nibrs_crime_against_category
7,other (includes drug and sex offenses),999,999,not reportable to nibrs,b,not_a_crime


IMPORTANT NOTE - Currently the regex misses the nibrs_crime_against_category columns frequent not_a_crime entry

## 3. Composition of the `other` crime category

The production preparation layer currently maps the source category
`all other` into the dashboard category:

`other (includes drug and sex offenses)`

This section breaks that category down by subcategory and NIBRS classification.

In [ ]:
OTHER_CATEGORY = (
    "other (includes drug and sex offenses)"
)

other = valid_time[
    clean_string(
        valid_time[CATEGORY_COLUMN]
    ).eq(OTHER_CATEGORY)
].copy()

all_unique_offenses = (
    valid_time[EVENT_ID_COLUMN].nunique()
)

other_unique_offenses = (
    other[EVENT_ID_COLUMN].nunique()
)

print(
    f"Total unique offenses: {all_unique_offenses:,}"
)

print(
    f"Other-category offenses: {other_unique_offenses:,}"
)

print(
    f"Share of all offenses: "
    f"{100 * other_unique_offenses / all_unique_offenses:.2f}%"
)


# -------------------------------------------------------------------
# Primary breakdown: offense subcategory.
# -------------------------------------------------------------------

other_by_subcategory = (
    other
    .groupby(
        SUB_CATEGORY_COLUMN,
        dropna=False,
    )
    .agg(
        unique_offenses=(
            EVENT_ID_COLUMN,
            "nunique",
        ),
        unique_reports=(
            ROW_ID_COLUMN,
            "nunique",
        ),
    )
    .reset_index()
    .sort_values(
        "unique_offenses",
        ascending=False,
    )
)

other_by_subcategory["share_of_other_pct"] = (
    100
    * other_by_subcategory["unique_offenses"]
    / other_unique_offenses
)

other_by_subcategory["share_of_all_crime_pct"] = (
    100
    * other_by_subcategory["unique_offenses"]
    / all_unique_offenses
)

display(other_by_subcategory)

Total unique offenses: 76,139
Other-category offenses: 36,394
Share of all offenses: 47.80%


,offense_sub_category,unique_offenses,unique_reports,share_of_other_pct,share_of_all_crime_pct
0,999,9088,7954,24.971149,11.936064
3,assault offenses,6697,6639,18.401385,8.795755
15,"property offenses (includes stolen, destruction)",6287,6280,17.274826,8.257266
6,extortion/fraud/forgery/bribery (includes bad checks),3276,3079,9.001484,4.302657
1,all other,2615,2583,7.185250,3.434508
12,narcotic violations (includes drug equip.),2353,2036,6.465351,3.090400
18,trespass,1452,1451,3.989669,1.907038
20,violation of no contact order,1414,1414,3.885256,1.857130
5,dui,1139,1139,3.129637,1.495948
21,weapon law violation,992,992,2.725724,1.302880


In [ ]:
other_crosswalk_columns = [
    SUB_CATEGORY_COLUMN,
    "nibrs_offense_code",
    "nibrs_offense_code_description",
    "nibrs_group_a_b",
    "nibrs_crime_against_category",
]

other_crosswalk_columns = [
    column
    for column in other_crosswalk_columns
    if column in other.columns
]

other_crosswalk = (
    other
    .groupby(
        other_crosswalk_columns,
        dropna=False,
    )
    .agg(
        unique_offenses=(
            EVENT_ID_COLUMN,
            "nunique",
        ),
        unique_reports=(
            ROW_ID_COLUMN,
            "nunique",
        ),
    )
    .reset_index()
    .sort_values(
        "unique_offenses",
        ascending=False,
    )
)

other_crosswalk["share_of_other_pct"] = (
    100
    * other_crosswalk["unique_offenses"]
    / other_unique_offenses
)

display(other_crosswalk)

,offense_sub_category,nibrs_offense_code,nibrs_offense_code_description,nibrs_group_a_b,nibrs_crime_against_category,unique_offenses,unique_reports,share_of_other_pct
0,999,999,not reportable to nibrs,b,not_a_crime,9088,7954,24.971149
33,"property offenses (includes stolen, destruction)",290,destruction/damage/vandalism of property,a,property,5742,5742,15.777326
3,assault offenses,13b,simple assault,a,person,4607,4607,12.658680
1,all other,90z,all other offenses,b,any,2615,2583,7.185250
4,assault offenses,13c,intimidation,a,person,2090,2090,5.742705
28,narcotic violations (includes drug equip.),35a,drug/narcotic violations,a,society,1811,1811,4.976095
41,trespass,90j,trespass of real,b,society,1452,1451,3.989669
43,violation of no contact order,500,violation of no contact orders,a,person,1414,1414,3.885256
7,dui,90d,driving under the influence,b,society,1139,1139,3.129637
44,weapon law violation,520,weapon law violations,a,society,992,992,2.725724


## 4. Classification decisions for `other`

Each distinct subcategory/NIBRS-code combination should receive one of:

- `retain`
- `reclassify`
- `exclude`
- `review`

Decisions should be justified from the dataset semantics rather than inferred
solely from a keyword.

This table is the audit trail for future production classification rules.

In [ ]:
decision_key_columns = [
    SUB_CATEGORY_COLUMN,
    "nibrs_offense_code",
    "nibrs_offense_code_description",
]

decision_key_columns = [
    column
    for column in decision_key_columns
    if column in other.columns
]

other_decision_table = (
    other
    .groupby(
        decision_key_columns,
        dropna=False,
    )
    .agg(
        unique_offenses=(
            EVENT_ID_COLUMN,
            "nunique",
        ),
    )
    .reset_index()
    .sort_values(
        "unique_offenses",
        ascending=False,
    )
)

other_decision_table["share_of_other_pct"] = (
    100
    * other_decision_table["unique_offenses"]
    / other_unique_offenses
)


# Evidence flags only.
# These are NOT automatic exclusion decisions.

combined_text = (
    other_decision_table
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.lower()
)

other_decision_table["possible_noncrime_flag"] = (
    combined_text.str.contains(
        NON_CRIME_PATTERN,
        regex=True,
        na=False,
    )
)

other_decision_table["contains_999_flag"] = (
    combined_text.str.contains(
        r"(?<!\d)999(?!\d)",
        regex=True,
        na=False,
    )
)

other_decision_table["action"] = "review"
other_decision_table["target_category"] = pd.NA
other_decision_table["decision_reason"] = pd.NA

display(other_decision_table)

,offense_sub_category,nibrs_offense_code,nibrs_offense_code_description,unique_offenses,share_of_other_pct,possible_noncrime_flag,contains_999_flag,action,target_category,decision_reason
0,999,999,not reportable to nibrs,9088,24.971149,False,True,review,<NA>,<NA>
33,"property offenses (includes stolen, destruction)",290,destruction/damage/vandalism of property,5742,15.777326,False,False,review,<NA>,<NA>
3,assault offenses,13b,simple assault,4607,12.658680,False,False,review,<NA>,<NA>
1,all other,90z,all other offenses,2615,7.185250,False,False,review,<NA>,<NA>
4,assault offenses,13c,intimidation,2090,5.742705,False,False,review,<NA>,<NA>
28,narcotic violations (includes drug equip.),35a,drug/narcotic violations,1811,4.976095,False,False,review,<NA>,<NA>
41,trespass,90j,trespass of real,1452,3.989669,False,False,review,<NA>,<NA>
43,violation of no contact order,500,violation of no contact orders,1414,3.885256,False,False,review,<NA>,<NA>
7,dui,90d,driving under the influence,1139,3.129637,False,False,review,<NA>,<NA>
44,weapon law violation,520,weapon law violations,992,2.725724,False,False,review,<NA>,<NA>


In [ ]:
# -------------------------------------------------------------------
# Fill this mapping only AFTER reviewing the evidence above.
#
# Key:
# (
#     offense_sub_category,
#     nibrs_offense_code,
# )
#
# Valid actions:
#     retain
#     reclassify
#     exclude
#     review
#
# Example structure ONLY:
#
# OTHER_DECISIONS = {
#     ("some subcategory", "some code"): {
#         "action": "reclassify",
#         "target_category": "violent crime",
#         "reason": "Documented reason here",
#     },
# }
# -------------------------------------------------------------------

OTHER_DECISIONS = {
    (
        "999",
        "999",
    ): {
        "action": "exclude",
        "target_category": pd.NA,
        "reason": (
            "Seattle source data uses 999 as a placeholder / "
            "'not a crime' classification, so it should be excluded "
            "from crime analysis."
        ),
    },

    (
        "property offenses (includes stolen, destruction)",
        "290",
    ): {
        "action": "reclassify",
        "target_category": "property crime",
        "reason": (
            "Destruction, Damage, or Vandalism of Property is "
            "classified as a Crime Against Property."
        ),
    },

    (
        "assault offenses",
        "13b",
    ): {
        "action": "reclassify",
        "target_category": "violent / person crime",
        "reason": (
            "NIBRS classifies Simple Assault as a Crime Against Person."
        ),
    },

    (
        "all other",
        "90z",
    ): {
        "action": "retain",
        "target_category": "crimes against society / other",
        "reason": (
            "All Other Offenses is a catch-all NIBRS category for "
            "offenses not otherwise classified and is retained in the "
            "Crimes Against Society / Other dashboard category."
        ),
    },

    (
        "assault offenses",
        "13c",
    ): {
        "action": "reclassify",
        "target_category": "violent / person crime",
        "reason": (
            "NIBRS classifies Intimidation as a Crime Against Person."
        ),
    },

    (
        "narcotic violations (includes drug equip.)",
        "35a",
    ): {
        "action": "retain",
        "target_category": "crimes against society / other",
        "reason": (
            "NIBRS classifies Drug/Narcotic Violations "
            "as a Crime Against Society."
        ),
    },

    (
        "trespass",
        "90j",
    ): {
        "action": "retain",
        "target_category": "crimes against society / other",
        "reason": (
            "NIBRS classifies Trespass of Real Property "
            "as a Crime Against Society."
        ),
    },

    (
        "violation of no contact order",
        "500",
    ): {
        "action": "retain",
        "target_category": "crimes against society / other",
        "reason": (
            "Violation of a No Contact Order does not represent a "
            "property offense and is retained in the Crimes Against "
            "Society / Other dashboard category."
        ),
    },

    (
        "dui",
        "90d",
    ): {
        "action": "retain",
        "target_category": "crimes against society / other",
        "reason": (
            "NIBRS classifies Driving Under the Influence "
            "as a Crime Against Society."
        ),
    },

    (
        "weapon law violation",
        "520",
    ): {
        "action": "retain",
        "target_category": "crimes against society / other",
        "reason": (
            "NIBRS classifies Weapon Law Violations "
            "as Crimes Against Society."
        ),
    },

    (
        "extortion/fraud/forgery/bribery (includes bad checks)",
        "26b",
    ): {
        "action": "reclassify",
        "target_category": "property crime",
        "reason": (
            "NIBRS classifies Credit Card / Automated Teller Machine "
            "Fraud as a Crime Against Property."
        ),
    },

    (
        "extortion/fraud/forgery/bribery (includes bad checks)",
        "26f",
    ): {
        "action": "reclassify",
        "target_category": "property crime",
        "reason": (
            "NIBRS classifies Identity Theft as a Crime Against Property."
        ),
    },

    (
        "extortion/fraud/forgery/bribery (includes bad checks)",
        "26a",
    ): {
        "action": "reclassify",
        "target_category": "property crime",
        "reason": (
            "NIBRS classifies False Pretenses / Swindle / Confidence "
            "Game as a Crime Against Property."
        ),
    },

    (
        "property offenses (includes stolen, destruction)",
        "280",
    ): {
        "action": "reclassify",
        "target_category": "property crime",
        "reason": (
            "NIBRS classifies Stolen Property Offenses "
            "as Crimes Against Property."
        ),
    },

    (
        "narcotic violations (includes drug equip.)",
        "35b",
    ): {
        "action": "retain",
        "target_category": "crimes against society / other",
        "reason": (
            "NIBRS classifies Drug Equipment Violations "
            "as a Crime Against Society."
        ),
    },

    (
        "extortion/fraud/forgery/bribery (includes bad checks)",
        "26c",
    ): {
        "action": "reclassify",
        "target_category": "property crime",
        "reason": (
            "NIBRS classifies Impersonation as a Crime Against Property."
        ),
    },

    (
        "extortion/fraud/forgery/bribery (includes bad checks)",
        "26e",
    ): {
        "action": "reclassify",
        "target_category": "property crime",
        "reason": (
            "NIBRS classifies Wire Fraud as a Crime Against Property."
        ),
    },

    (
        "disorderly conduct & vagrancy violations",
        "90c",
    ): {
        "action": "retain",
        "target_category": "crimes against society / other",
        "reason": (
            "NIBRS classifies Disorderly Conduct "
            "as a Crime Against Society."
        ),
    },

    (
        "kidnapping/abduction",
        "100",
    ): {
        "action": "reclassify",
        "target_category": "violent / person crime",
        "reason": (
            "NIBRS classifies Kidnapping / Abduction "
            "as a Crime Against Person."
        ),
    },

    (
        "sex offenses",
        "11d",
    ): {
        "action": "reclassify",
        "target_category": "violent / person crime",
        "reason": (
            "NIBRS classifies Criminal Sexual Contact / Fondling "
            "as a Crime Against Person."
        ),
    },

    (
        "extortion/fraud/forgery/bribery (includes bad checks)",
        "210",
    ): {
        "action": "reclassify",
        "target_category": "property crime",
        "reason": (
            "NIBRS classifies Extortion / Blackmail "
            "as a Crime Against Property."
        ),
    },

    (
        "extortion/fraud/forgery/bribery (includes bad checks)",
        "250",
    ): {
        "action": "reclassify",
        "target_category": "property crime",
        "reason": (
            "NIBRS classifies Counterfeiting / Forgery "
            "as a Crime Against Property."
        ),
    },

    (
        "extortion/fraud/forgery/bribery (includes bad checks)",
        "26g",
    ): {
        "action": "reclassify",
        "target_category": "property crime",
        "reason": (
            "NIBRS classifies Hacking / Computer Invasion "
            "as a Crime Against Property."
        ),
    },

    (
        "non-violent family offenses",
        "90f",
    ): {
        "action": "retain",
        "target_category": "crimes against society / other",
        "reason": (
            "NIBRS classifies Family Offenses, Nonviolent "
            "as Crimes Against Society."
        ),
    },

    (
        "pornography",
        "370",
    ): {
        "action": "retain",
        "target_category": "crimes against society / other",
        "reason": (
            "NIBRS classifies Pornography / Obscene Material "
            "as a Crime Against Society."
        ),
    },

    (
        "animal cruelty",
        "720",
    ): {
        "action": "retain",
        "target_category": "crimes against society / other",
        "reason": (
            "NIBRS classifies Animal Cruelty "
            "as a Crime Against Society."
        ),
    },

    (
        "unknown",
        "-",
    ): {
        "action": "exclude",
        "target_category": pd.NA,
        "reason": (
            "The record has no usable offense classification and "
            "should not contribute to classified crime analysis."
        ),
    },

    (
        "extortion/fraud/forgery/bribery (includes bad checks)",
        "270",
    ): {
        "action": "reclassify",
        "target_category": "property crime",
        "reason": (
            "NIBRS classifies Embezzlement as a Crime Against Property."
        ),
    },

    (
        "liquor law violations & drunkenness",
        "90g",
    ): {
        "action": "retain",
        "target_category": "crimes against society / other",
        "reason": (
            "NIBRS classifies Liquor Law Violations "
            "as a Crime Against Society."
        ),
    },

    (
        "prostitution offenses",
        "40c",
    ): {
        "action": "retain",
        "target_category": "crimes against society / other",
        "reason": (
            "NIBRS classifies Purchasing Prostitution "
            "as a Crime Against Society."
        ),
    },

    (
        "human trafficking",
        "64a",
    ): {
        "action": "reclassify",
        "target_category": "violent / person crime",
        "reason": (
            "NIBRS classifies Human Trafficking, Commercial Sex Acts "
            "as a Crime Against Person."
        ),
    },

    (
        "disorderly conduct & vagrancy violations",
        "90b",
    ): {
        "action": "retain",
        "target_category": "crimes against society / other",
        "reason": (
            "NIBRS classifies Curfew / Loitering / Vagrancy Violations "
            "as Crimes Against Society."
        ),
    },

    (
        "prostitution offenses",
        "40b",
    ): {
        "action": "retain",
        "target_category": "crimes against society / other",
        "reason": (
            "NIBRS classifies Assisting or Promoting Prostitution "
            "as a Crime Against Society."
        ),
    },

    (
        "extortion/fraud/forgery/bribery (includes bad checks)",
        "90a",
    ): {
        "action": "reclassify",
        "target_category": "property crime",
        "reason": (
            "Bad Checks represents a property/economic offense. "
            "Although 90A is a historical NIBRS code, it is most "
            "appropriately grouped with Property Crime for the "
            "dashboard's analytical categories."
        ),
    },

    (
        "sex offenses",
        "36b",
    ): {
        "action": "reclassify",
        "target_category": "violent / person crime",
        "reason": (
            "NIBRS classifies Statutory Rape "
            "as a Crime Against Person."
        ),
    },

    (
        "prostitution offenses",
        "40a",
    ): {
        "action": "retain",
        "target_category": "crimes against society / other",
        "reason": (
            "NIBRS classifies Prostitution "
            "as a Crime Against Society."
        ),
    },

    (
        "liquor law violations & drunkenness",
        "90e",
    ): {
        "action": "retain",
        "target_category": "crimes against society / other",
        "reason": (
            "NIBRS classifies Drunkenness "
            "as a Crime Against Society."
        ),
    },

    (
        "extortion/fraud/forgery/bribery (includes bad checks)",
        "26d",
    ): {
        "action": "reclassify",
        "target_category": "property crime",
        "reason": (
            "NIBRS classifies Welfare Fraud as a Crime Against Property."
        ),
    },

    (
        "justifiable homicide",
        "09c",
    ): {
        "action": "exclude",
        "target_category": pd.NA,
        "reason": (
            "NIBRS identifies Justifiable Homicide as 'Not a Crime', "
            "so it should be excluded from crime totals and analysis."
        ),
    },

    (
        "sex offenses",
        "90h",
    ): {
        "action": "retain",
        "target_category": "crimes against society / other",
        "reason": (
            "NIBRS classifies Peeping Tom "
            "as a Crime Against Society."
        ),
    },

    (
        "gambling offenses",
        "39a",
    ): {
        "action": "retain",
        "target_category": "crimes against society / other",
        "reason": (
            "NIBRS classifies Betting / Wagering "
            "as a Crime Against Society."
        ),
    },

    (
        "sex offenses",
        "36a",
    ): {
        "action": "reclassify",
        "target_category": "violent / person crime",
        "reason": (
            "NIBRS classifies Incest as a Crime Against Person."
        ),
    },

    (
        "gambling offenses",
        "39c",
    ): {
        "action": "retain",
        "target_category": "crimes against society / other",
        "reason": (
            "NIBRS classifies Operating / Promoting / Assisting "
            "Gambling as a Crime Against Society."
        ),
    },

    (
        "extortion/fraud/forgery/bribery (includes bad checks)",
        "510",
    ): {
        "action": "reclassify",
        "target_category": "property crime",
        "reason": (
            "NIBRS classifies Bribery as a Crime Against Property."
        ),
    },

    (
        "gambling offenses",
        "39b",
    ): {
        "action": "retain",
        "target_category": "crimes against society / other",
        "reason": (
            "NIBRS classifies Operating / Promoting / Assisting "
            "Gambling as a Crime Against Society."
        ),
    },
}

ALLOWED_ACTIONS = {
    "retain",
    "reclassify",
    "exclude",
    "review",
}


def apply_other_decision(row):
    key = (
        row.get(SUB_CATEGORY_COLUMN),
        row.get("nibrs_offense_code"),
    )

    decision = OTHER_DECISIONS.get(key)

    if decision is None:
        return pd.Series(
            {
                "action": "review",
                "target_category": pd.NA,
                "decision_reason": pd.NA,
            }
        )

    action = decision["action"]

    if action not in ALLOWED_ACTIONS:
        raise ValueError(
            f"Invalid action {action!r} for {key}"
        )

    return pd.Series(
        {
            "action": action,
            "target_category": decision.get(
                "target_category"
            ),
            "decision_reason": decision.get(
                "reason"
            ),
        }
    )


decision_results = (
    other_decision_table
    .drop(
        columns=[
            "action",
            "target_category",
            "decision_reason",
        ]
    )
    .copy()
)

decision_results[
    [
        "action",
        "target_category",
        "decision_reason",
    ]
] = decision_results.apply(
    apply_other_decision,
    axis=1,
)

display(decision_results)

,offense_sub_category,nibrs_offense_code,nibrs_offense_code_description,unique_offenses,share_of_other_pct,possible_noncrime_flag,contains_999_flag,action,target_category,decision_reason
0,999,999,not reportable to nibrs,9088,24.971149,False,True,exclude,NaN,"Seattle source data uses 999 as a placeholder / 'not a crime' classification, so it should be ex..."
33,"property offenses (includes stolen, destruction)",290,destruction/damage/vandalism of property,5742,15.777326,False,False,reclassify,property crime,"Destruction, Damage, or Vandalism of Property is classified as a Crime Against Property."
3,assault offenses,13b,simple assault,4607,12.658680,False,False,reclassify,violent / person crime,NIBRS classifies Simple Assault as a Crime Against Person.
1,all other,90z,all other offenses,2615,7.185250,False,False,retain,crimes against society / other,All Other Offenses is a catch-all NIBRS category for offenses not otherwise classified and is re...
4,assault offenses,13c,intimidation,2090,5.742705,False,False,reclassify,violent / person crime,NIBRS classifies Intimidation as a Crime Against Person.
28,narcotic violations (includes drug equip.),35a,drug/narcotic violations,1811,4.976095,False,False,retain,crimes against society / other,NIBRS classifies Drug/Narcotic Violations as a Crime Against Society.
41,trespass,90j,trespass of real,1452,3.989669,False,False,retain,crimes against society / other,NIBRS classifies Trespass of Real Property as a Crime Against Society.
43,violation of no contact order,500,violation of no contact orders,1414,3.885256,False,False,retain,crimes against society / other,Violation of a No Contact Order does not represent a property offense and is retained in the Cri...
7,dui,90d,driving under the influence,1139,3.129637,False,False,retain,crimes against society / other,NIBRS classifies Driving Under the Influence as a Crime Against Society.
44,weapon law violation,520,weapon law violations,992,2.725724,False,False,retain,crimes against society / other,NIBRS classifies Weapon Law Violations as Crimes Against Society.


In [ ]:
decision_summary = (
    decision_results
    .groupby(
        "action",
        dropna=False,
    )
    .agg(
        unique_offenses=(
            "unique_offenses",
            "sum",
        ),
    )
    .reset_index()
)

decision_summary["share_of_other_pct"] = (
    100
    * decision_summary["unique_offenses"]
    / other_unique_offenses
)

display(decision_summary)


unresolved = decision_results[
    decision_results["action"].eq("review")
]

print(
    f"Still requiring review: "
    f"{unresolved['unique_offenses'].sum():,} offense(s)"
)

if not unresolved.empty:
    display(unresolved)

unresolved.to_clipboard()

,action,unique_offenses,share_of_other_pct
0,exclude,9135,25.100291
1,reclassify,16736,45.985602
2,retain,10523,28.914107


Still requiring review: 0 offense(s)


## 5. Broader KPI data-quality audit

Future KPI cards will depend on consistent counting semantics.

This section checks for:

- missing identifiers and timestamps
- missing classification values
- unknown/placeholder neighborhood values
- missing/placeholder NIBRS codes
- duplicate offense IDs
- offense IDs appearing on multiple dates
- offense IDs changing category or subcategory
- multiple offenses attached to one report
- invalid coordinates
- reporting timestamps earlier than offense timestamps
- unexpected top-level crime categories

Some findings may be legitimate data semantics rather than errors. The purpose
is to identify anything that must be explicitly accounted for before KPI logic
is finalized.

In [ ]:
audit = crime.copy()

audit[EVENT_ID_COLUMN] = clean_string(
    audit[EVENT_ID_COLUMN]
)

audit[ROW_ID_COLUMN] = clean_string(
    audit[ROW_ID_COLUMN]
)

audit[TIME_COLUMN] = pd.to_datetime(
    audit[TIME_COLUMN],
    errors="coerce",
)

audit[REPORT_TIME_COLUMN] = pd.to_datetime(
    audit[REPORT_TIME_COLUMN],
    errors="coerce",
)

event_id_valid = valid_string_mask(
    audit[EVENT_ID_COLUMN]
)

total_valid_ids = (
    audit.loc[
        event_id_valid,
        EVENT_ID_COLUMN,
    ]
    .nunique()
)


def make_issue(
    issue,
    mask,
    why_it_matters,
    classification="Potential distortion",
):
    mask = mask.fillna(False)

    rows = int(mask.sum())

    unique_offenses = (
        audit.loc[
            mask & event_id_valid,
            EVENT_ID_COLUMN,
        ]
        .nunique()
    )

    share = (
        100 * unique_offenses / total_valid_ids
        if total_valid_ids
        else np.nan
    )

    return {
        "issue": issue,
        "classification": classification,
        "rows_affected": rows,
        "unique_offenses_affected": unique_offenses,
        "share_of_valid_offenses_pct": round(
            share,
            3,
        ),
        "why_it_matters": why_it_matters,
    }


issues = []


# -------------------------------------------------------------------
# Missing core fields
# -------------------------------------------------------------------

missing_event_id = ~event_id_valid

missing_offense_date = audit[TIME_COLUMN].isna()

missing_report_number = ~valid_string_mask(
    audit[ROW_ID_COLUMN]
)

missing_category = ~valid_string_mask(
    audit[CATEGORY_COLUMN]
)

missing_subcategory = ~valid_string_mask(
    audit[SUB_CATEGORY_COLUMN]
)

issues.append(
    make_issue(
        "Missing/invalid offense ID",
        missing_event_id,
        "Cannot reliably count a unique offense.",
    )
)

issues.append(
    make_issue(
        "Missing offense date",
        missing_offense_date,
        "Cannot place the offense in a time period.",
    )
)

issues.append(
    make_issue(
        "Missing/invalid report number",
        missing_report_number,
        "Affects report-level versus offense-level comparisons.",
    )
)

issues.append(
    make_issue(
        "Missing/placeholder crime category",
        missing_category,
        "Could disappear from category-specific KPIs.",
    )
)

issues.append(
    make_issue(
        "Missing/placeholder crime subcategory",
        missing_subcategory,
        "Could distort subtype breakdowns.",
    )
)


# -------------------------------------------------------------------
# Neighborhood quality
# -------------------------------------------------------------------

if "mcpp_neighborhood" in valid_time.columns:
    bad_analytic_neighborhood_ids = set(
        clean_string(
            valid_time.loc[
                ~valid_string_mask(
                    valid_time["mcpp_neighborhood"]
                ),
                EVENT_ID_COLUMN,
            ]
        )
        .dropna()
        .tolist()
    )

    bad_analytic_neighborhood = (
        audit[EVENT_ID_COLUMN].isin(
            bad_analytic_neighborhood_ids
        )
    )

    issues.append(
        make_issue(
            "Missing/placeholder analytical neighborhood",
            bad_analytic_neighborhood,
            (
                "These offenses can remain in citywide totals but "
                "cannot be assigned cleanly to neighborhood rankings."
            ),
            classification="Known geographic limitation",
        )
    )


# -------------------------------------------------------------------
# NIBRS code quality
# -------------------------------------------------------------------

if "nibrs_offense_code" in audit.columns:
    nibrs_code = clean_string(
        audit["nibrs_offense_code"]
    )

    missing_nibrs_code = ~valid_string_mask(
        audit["nibrs_offense_code"]
    )

    nibrs_999 = nibrs_code.str.contains(
        r"(?<!\d)999(?!\d)",
        regex=True,
        na=False,
    )

    issues.append(
        make_issue(
            "Missing/placeholder NIBRS offense code",
            missing_nibrs_code,
            "May prevent reliable offense-code classification.",
        )
    )

    issues.append(
        make_issue(
            "NIBRS offense code contains 999",
            nibrs_999,
            "Requires semantic review before inclusion/exclusion rules are finalized.",
            classification="Requires semantic review",
        )
    )


# -------------------------------------------------------------------
# Unexpected top-level categories
# -------------------------------------------------------------------

category_clean = clean_string(
    audit[CATEGORY_COLUMN]
)

unexpected_category = (
    valid_string_mask(
        audit[CATEGORY_COLUMN]
    )
    & ~category_clean.isin(
        TARGET_CRIME_CATEGORIES
    )
)

issues.append(
    make_issue(
        "Unexpected top-level crime category",
        unexpected_category,
        "Could be silently excluded by dashboard category filters.",
    )
)


# -------------------------------------------------------------------
# Coordinate quality
# -------------------------------------------------------------------

latitude = pd.to_numeric(
    audit[LAT_COL],
    errors="coerce",
)

longitude = pd.to_numeric(
    audit[LON_COL],
    errors="coerce",
)

valid_coordinates = (
    latitude.between(47.45, 47.80)
    & longitude.between(-122.45, -122.20)
)

issues.append(
    make_issue(
        "Missing/out-of-range Seattle coordinates",
        ~valid_coordinates,
        (
            "Should affect mapping only, not citywide offense KPIs. "
            "Useful for detecting accidental map-driven exclusions."
        ),
        classification="Known geographic limitation",
    )
)


# -------------------------------------------------------------------
# Reporting timestamp behavior
# -------------------------------------------------------------------

negative_reporting_lag = (
    audit[TIME_COLUMN].notna()
    & audit[REPORT_TIME_COLUMN].notna()
    & (
        audit[REPORT_TIME_COLUMN]
        < audit[TIME_COLUMN]
    )
)

issues.append(
    make_issue(
        "Report timestamp earlier than offense timestamp",
        negative_reporting_lag,
        "May represent legitimate semantics or timestamp inconsistency; inspect before using reporting lag.",
        classification="Requires semantic review",
    )
)


# -------------------------------------------------------------------
# Duplicate offense IDs / conflicts
# -------------------------------------------------------------------

valid_id_rows = audit[
    event_id_valid
].copy()

event_group = valid_id_rows.groupby(
    EVENT_ID_COLUMN
)

event_row_counts = event_group.size()

duplicate_ids = set(
    event_row_counts[
        event_row_counts > 1
    ].index
)

duplicate_id_mask = audit[
    EVENT_ID_COLUMN
].isin(duplicate_ids)

issues.append(
    make_issue(
        "Offense ID appears on multiple rows",
        duplicate_id_mask,
        (
            "Not necessarily incorrect, but KPI code must use "
            "unique offense IDs rather than raw row counts."
        ),
        classification="Counting-semantic risk",
    )
)


# Multiple offense dates for same offense ID
date_counts_by_event = (
    valid_id_rows
    .assign(
        normalized_date=valid_id_rows[
            TIME_COLUMN
        ].dt.normalize()
    )
    .groupby(EVENT_ID_COLUMN)[
        "normalized_date"
    ]
    .nunique(dropna=True)
)

multi_date_ids = set(
    date_counts_by_event[
        date_counts_by_event > 1
    ].index
)

issues.append(
    make_issue(
        "Same offense ID appears on multiple offense dates",
        audit[EVENT_ID_COLUMN].isin(
            multi_date_ids
        ),
        (
            "Could cause one offense to be counted on more than one day "
            "even when daily aggregation uses nunique within each day."
        ),
    )
)


# Multiple categories for same offense ID
category_counts_by_event = (
    valid_id_rows
    .groupby(EVENT_ID_COLUMN)[
        CATEGORY_COLUMN
    ]
    .nunique(dropna=True)
)

multi_category_ids = set(
    category_counts_by_event[
        category_counts_by_event > 1
    ].index
)

issues.append(
    make_issue(
        "Same offense ID has multiple top-level categories",
        audit[EVENT_ID_COLUMN].isin(
            multi_category_ids
        ),
        "Could create ambiguous category-level KPI attribution.",
    )
)


# Multiple subcategories for same offense ID
subcategory_counts_by_event = (
    valid_id_rows
    .groupby(EVENT_ID_COLUMN)[
        SUB_CATEGORY_COLUMN
    ]
    .nunique(dropna=True)
)

multi_subcategory_ids = set(
    subcategory_counts_by_event[
        subcategory_counts_by_event > 1
    ].index
)

issues.append(
    make_issue(
        "Same offense ID has multiple subcategories",
        audit[EVENT_ID_COLUMN].isin(
            multi_subcategory_ids
        ),
        "Could create ambiguous subtype attribution.",
    )
)


# -------------------------------------------------------------------
# Reports containing multiple offenses
# -------------------------------------------------------------------

valid_report_rows = audit[
    valid_string_mask(
        audit[ROW_ID_COLUMN]
    )
    & event_id_valid
].copy()

offenses_per_report = (
    valid_report_rows
    .groupby(ROW_ID_COLUMN)[
        EVENT_ID_COLUMN
    ]
    .nunique()
)

multi_offense_reports = set(
    offenses_per_report[
        offenses_per_report > 1
    ].index
)

multi_offense_report_mask = audit[
    ROW_ID_COLUMN
].isin(
    multi_offense_reports
)

issues.append(
    make_issue(
        "Report contains multiple unique offenses",
        multi_offense_report_mask,
        (
            "Expected in many crime datasets, but establishes that "
            "'offenses' and 'reports/incidents' are different KPI denominators."
        ),
        classification="Counting-semantic distinction",
    )
)


# -------------------------------------------------------------------
# Final audit table
# -------------------------------------------------------------------

audit_summary = (
    pd.DataFrame(issues)
    .sort_values(
        [
            "classification",
            "unique_offenses_affected",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)

display(audit_summary)

,issue,classification,rows_affected,unique_offenses_affected,share_of_valid_offenses_pct,why_it_matters
0,Report contains multiple unique offenses,Counting-semantic distinction,21268,21268,27.933,"Expected in many crime datasets, but establishes that 'offenses' and 'reports/incidents' are dif..."
1,Offense ID appears on multiple rows,Counting-semantic risk,0,0,0.000,"Not necessarily incorrect, but KPI code must use unique offense IDs rather than raw row counts."
2,Missing/out-of-range Seattle coordinates,Known geographic limitation,13322,13322,17.497,"Should affect mapping only, not citywide offense KPIs. Useful for detecting accidental map-drive..."
3,Missing/placeholder analytical neighborhood,Known geographic limitation,1218,1218,1.600,These offenses can remain in citywide totals but cannot be assigned cleanly to neighborhood rank...
4,Missing/placeholder crime subcategory,Potential distortion,44,44,0.058,Could distort subtype breakdowns.
5,Missing/placeholder NIBRS offense code,Potential distortion,44,44,0.058,May prevent reliable offense-code classification.
6,Missing/invalid offense ID,Potential distortion,0,0,0.000,Cannot reliably count a unique offense.
7,Missing offense date,Potential distortion,0,0,0.000,Cannot place the offense in a time period.
8,Missing/invalid report number,Potential distortion,0,0,0.000,Affects report-level versus offense-level comparisons.
9,Missing/placeholder crime category,Potential distortion,0,0,0.000,Could disappear from category-specific KPIs.


In [ ]:
print("Offense IDs appearing on multiple dates")
print("-----------------------------------------")

multi_date_detail = (
    valid_id_rows[
        valid_id_rows[
            EVENT_ID_COLUMN
        ].isin(multi_date_ids)
    ]
    [
        [
            EVENT_ID_COLUMN,
            TIME_COLUMN,
            CATEGORY_COLUMN,
            SUB_CATEGORY_COLUMN,
        ]
    ]
    .sort_values(
        [
            EVENT_ID_COLUMN,
            TIME_COLUMN,
        ]
    )
)

display(multi_date_detail.head(100))


print()
print("Unexpected top-level crime categories")
print("-------------------------------------")

unexpected_category_summary = (
    audit.loc[
        unexpected_category
    ]
    .groupby(
        CATEGORY_COLUMN,
        dropna=False,
    )[EVENT_ID_COLUMN]
    .nunique()
    .sort_values(
        ascending=False
    )
    .rename("unique_offenses")
    .reset_index()
)

display(unexpected_category_summary)


print()
print("Reports containing multiple offenses")
print("------------------------------------")

display(
    offenses_per_report
    .value_counts()
    .sort_index()
    .rename_axis(
        "unique_offenses_per_report"
    )
    .rename(
        "report_count"
    )
    .reset_index()
)

Offense IDs appearing on multiple dates
-----------------------------------------


,offense_id,offense_date,offense_category,offense_sub_category



Unexpected top-level crime categories
-------------------------------------


,offense_category,unique_offenses



Reports containing multiple offenses
------------------------------------


,unique_offenses_per_report,report_count
0,1,54871
1,2,6796
2,3,1692
3,4,403
4,5,115
5,6,38
6,7,17
7,8,2
8,9,3
9,11,1


In [ ]:
nibrs_columns = [
    "nibrs_offense_code",
    "nibrs_offense_code_description",
    "nibrs_group_a_b",
    "nibrs_crime_against_category",
]

nibrs_columns = [
    column
    for column in nibrs_columns
    if column in audit.columns
]

if nibrs_columns:
    nibrs_quality = (
        audit
        .groupby(
            nibrs_columns,
            dropna=False,
        )
        .agg(
            unique_offenses=(
                EVENT_ID_COLUMN,
                "nunique",
            ),
        )
        .reset_index()
        .sort_values(
            "unique_offenses",
            ascending=False,
        )
    )

    display(nibrs_quality)

,nibrs_offense_code,nibrs_offense_code_description,nibrs_group_a_b,nibrs_crime_against_category,unique_offenses
61,999,not reportable to nibrs,b,not_a_crime,9088
20,23f,theft from motor vehicle,a,property,8412
14,220,burglary/breaking & entering,a,property,7555
34,290,destruction/damage/vandalism of property,a,property,5742
22,23h,all other larceny,a,property,5460
23,240,motor vehicle theft,a,property,4945
10,13b,simple assault,a,person,4607
17,23c,shoplifting,a,property,3160
9,13a,aggravated assault,a,person,3155
21,23g,theft of motor vehicle parts or accessories,a,property,2657


In [ ]:
print("v1.1 CRIME DATA QUALITY AUDIT STATUS")
print("=" * 60)

print()

print("1. Unmappable offense inclusion")
if mismatch_days.empty:
    print("   PASS — daily counts match valid_time in the chart window.")
else:
    print(
        f"   REVIEW — {len(mismatch_days):,} daily mismatch(es)."
    )

if expected_unmappable_ids == actual_unmappable_ids:
    print("   PASS — citywide filtering retains unmappable offenses.")
else:
    print("   REVIEW — citywide filtering loses unmappable offenses.")

if earliest_day_offenses:
    print(
        f"   NOTE — {earliest_day_offenses:,} offense(s) occur "
        "on the earliest available day, which should be reviewed "
        "against the current chart-window convention."
    )

print()

print("2. Code/subtype 999")
print(
    f"   {records_999[EVENT_ID_COLUMN].nunique():,} "
    "unique offense(s) associated with 999."
)
print("   Review the semantic crosswalk before making an exclusion rule.")

print()

print("3. Other category")
print(
    f"   {other_unique_offenses:,} unique offense(s), "
    f"{100 * other_unique_offenses / all_unique_offenses:.2f}% "
    "of analytically valid crime."
)

print()

print("4. Other-category decisions")
print(
    f"   {unresolved['unique_offenses'].sum():,} offense(s) "
    "currently remain marked REVIEW."
)

print()

print("5. Additional KPI risks")
potential_distortions = audit_summary[
    audit_summary["classification"].isin(
        {
            "Potential distortion",
            "Counting-semantic risk",
            "Requires semantic review",
        }
    )
]

print(
    f"   {len(potential_distortions):,} issue type(s) "
    "require review before KPI definitions are finalized."
)

display(potential_distortions)

v1.1 CRIME DATA QUALITY AUDIT STATUS

1. Unmappable offense inclusion
   PASS — daily counts match valid_time in the chart window.
   PASS — citywide filtering retains unmappable offenses.
   NOTE — 30 offense(s) occur on the earliest available day, which should be reviewed against the current chart-window convention.

2. Code/subtype 999
   9,088 unique offense(s) associated with 999.
   Review the semantic crosswalk before making an exclusion rule.

3. Other category
   36,394 unique offense(s), 47.80% of analytically valid crime.

4. Other-category decisions
   0 offense(s) currently remain marked REVIEW.

5. Additional KPI risks
   13 issue type(s) require review before KPI definitions are finalized.


,issue,classification,rows_affected,unique_offenses_affected,share_of_valid_offenses_pct,why_it_matters
1,Offense ID appears on multiple rows,Counting-semantic risk,0,0,0.000,"Not necessarily incorrect, but KPI code must use unique offense IDs rather than raw row counts."
4,Missing/placeholder crime subcategory,Potential distortion,44,44,0.058,Could distort subtype breakdowns.
5,Missing/placeholder NIBRS offense code,Potential distortion,44,44,0.058,May prevent reliable offense-code classification.
6,Missing/invalid offense ID,Potential distortion,0,0,0.000,Cannot reliably count a unique offense.
7,Missing offense date,Potential distortion,0,0,0.000,Cannot place the offense in a time period.
8,Missing/invalid report number,Potential distortion,0,0,0.000,Affects report-level versus offense-level comparisons.
9,Missing/placeholder crime category,Potential distortion,0,0,0.000,Could disappear from category-specific KPIs.
10,Unexpected top-level crime category,Potential distortion,0,0,0.000,Could be silently excluded by dashboard category filters.
11,Same offense ID appears on multiple offense dates,Potential distortion,0,0,0.000,Could cause one offense to be counted on more than one day even when daily aggregation uses nuni...
12,Same offense ID has multiple top-level categories,Potential distortion,0,0,0.000,Could create ambiguous category-level KPI attribution.
